# 04 - Eye movements

**Audience:** running the I-VT classifier and using the output.

Default thresholds match the I-VT literature (50 deg/s velocity, 100 ms
minimum fixation, 75 ms merge window — Salvucci & Goldberg 2000 plus
Olsson 2007 / Komogortsev 2010 follow-ups). The first section sweeps
the velocity threshold so you can pick a value justified for your data.

Most metrics in this notebook are reported **per phase**. Averaging
fixation duration across a free-look phase and a visual-search phase
smears two different behaviours into one number. Phase-aware splits
follow the four-phase battery shipped with `SampleExperiment` —
`FreeExploration`, `VisualSearch`, `CountingTask`, `ChangeDetection`.

## What you'll get

- Velocity-threshold sweep (visual signature of the I-VT elbow).
- Per-phase fixation, saccade, and scanpath statistics.
- Phase-coloured scanpath with fixation centroids sized by duration.
- K-coefficient: sliding-window K(t) and per-phase K computed
  against session-pooled (μ, σ) per Krejtz 2017.

## Why per-phase and pooled stats matter for K

Krejtz K is a difference of z-scores `K_i = z(a_i) − z(d_i)` averaged
over fixation/saccade pairs. If you compute (μ, σ) over the same set
you average, the result is identically zero — a mathematical truism,
not an empirical finding. Canonical Krejtz usage pools (μ, σ) over the
whole session and reports K either per-condition or as a sliding
K(t). This notebook does both.

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Convert combined gaze to angular yaw/pitch

The CSV stores combined gaze as a 3D unit vector. Convert to angular
yaw/pitch in degrees so the I-VT classifier and the K-coefficient
operate in their native units.

In [ ]:
df = ctx.data.df
ts = ctx.data.get_timestamps()
for c in ("combined_dir_x", "combined_dir_y", "combined_dir_z"):
    if c not in df.columns:
        raise RuntimeError(f"Missing required column: {c}")
dx = df["combined_dir_x"].to_numpy(dtype=float)
dy = df["combined_dir_y"].to_numpy(dtype=float)
dz = df["combined_dir_z"].to_numpy(dtype=float)
yaw = np.degrees(np.arctan2(dx, dz))
pitch = -np.degrees(np.arcsin(np.clip(dy, -1, 1)))

## 2. Velocity-threshold sweep

Vary the I-VT threshold and watch fixation count + median duration. The
literature 50 deg/s sits in the elbow of this curve for typical VR data.

In [ ]:
thresholds = [20, 30, 40, 50, 75, 100, 150]
rows = []
for th in thresholds:
    mv = ela.detect_eye_movements(yaw, pitch, ts, velocity_threshold=th)
    fixs_th = mv["fixations"]
    durs_ms = [f.duration * 1000 for f in fixs_th] if fixs_th else [0]
    rows.append({
        "threshold_deg_s": th,
        "n_fixations": len(fixs_th),
        "n_saccades": len(mv["saccades"]),
        "median_fix_ms": np.median(durs_ms),
        "p95_fix_ms": np.percentile(durs_ms, 95),
    })
sweep = pd.DataFrame(rows)
sweep

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(sweep["threshold_deg_s"], sweep["n_fixations"], marker="o")
axes[0].set_xlabel("velocity threshold (deg/s)")
axes[0].set_ylabel("# fixations")
axes[0].set_title("Fixation count vs. threshold")
axes[1].plot(sweep["threshold_deg_s"], sweep["median_fix_ms"], marker="o", color="darkorange")
axes[1].set_xlabel("velocity threshold (deg/s)")
axes[1].set_ylabel("median duration (ms)")
axes[1].set_title("Median fixation duration")
plt.tight_layout()
plt.show()

## 3. Detection at 50 deg/s + per-fixation phase assignment

Detect once over the whole recording, then assign each fixation to the
experiment phase that contains its start time. We use the start time
because phase boundaries are crisp; using the centroid time would
classify boundary fixations inconsistently.

In [ ]:
movements = ela.detect_eye_movements(yaw, pitch, ts, velocity_threshold=50.0)
fixs = movements["fixations"]
saccs = movements["saccades"]
n_pairs = min(len(fixs), len(saccs))
print(f"Whole recording: {len(fixs)} fixations, {len(saccs)} saccades ({n_pairs} fix/sac pairs)")

# Map each fixation/saccade to its phase via the start-time index.
phase_col = df["phase"].to_numpy() if ctx.data.has_column("phase") else None
def phase_of(start_time):
    if phase_col is None:
        return "Recording"
    idx = int(np.searchsorted(ts, start_time))
    idx = min(idx, len(phase_col) - 1)
    return phase_col[idx]

phase_per_fix = np.array([phase_of(f.start_time) for f in fixs])
phase_per_sac = np.array([phase_of(s.start_time) for s in saccs])

# Only keep the four canonical SampleExperiment phases for downstream
# splits. Idle / Instructions / Recording / Test rows are bookkeeping.
PHASES = ("FreeExploration", "VisualSearch", "CountingTask", "ChangeDetection")
observed_phases = [p for p in PHASES if (phase_per_fix == p).sum() >= 2]
print("Phases with >=2 fixations:", observed_phases or "[none — running on a non-SampleExperiment recording]")

## 4. Per-phase fixation and saccade summary

Each row covers fixations/saccades whose **start** falls inside that
phase. `mean_fix_ms` is the average fixation duration in milliseconds;
`mean_sac_deg` is the average saccade amplitude in degrees;
`scanpath_deg` is the sum of inter-sample angular displacements within
the phase (a coarse measure of how much the eyes moved).

In [ ]:
rows = []
for ph in observed_phases:
    fix_mask = phase_per_fix == ph
    sac_mask = phase_per_sac == ph
    fix_durs_ms = np.array([f.duration * 1000 for f in fixs if phase_of(f.start_time) == ph])
    sac_amps    = np.array([s.amplitude     for s in saccs if phase_of(s.start_time) == ph])
    if phase_col is not None:
        ph_mask = phase_col == ph
        ph_yaw = yaw[ph_mask]
        ph_pitch = pitch[ph_mask]
        scanpath_deg = float(np.sum(np.hypot(np.diff(ph_yaw), np.diff(ph_pitch)))) if len(ph_yaw) > 1 else float("nan")
        phase_duration_s = float(ph_mask.sum() / max(ctx.data.get_sample_rate(), 1.0))
    else:
        scanpath_deg = float("nan")
        phase_duration_s = float("nan")
    rows.append({
        "phase":         ph,
        "duration_s":    round(phase_duration_s, 1),
        "n_fix":         int(fix_mask.sum()),
        "n_sac":         int(sac_mask.sum()),
        "mean_fix_ms":   round(float(fix_durs_ms.mean()), 1) if len(fix_durs_ms) else None,
        "median_fix_ms": round(float(np.median(fix_durs_ms)), 1) if len(fix_durs_ms) else None,
        "mean_sac_deg":  round(float(sac_amps.mean()), 2)     if len(sac_amps)    else None,
        "scanpath_deg":  round(scanpath_deg, 0) if np.isfinite(scanpath_deg) else None,
    })
phase_table = pd.DataFrame(rows)
phase_table

## 5. Phase-coloured scanpath

Raw gaze in light grey; fixation centroids coloured by the phase that
contained them, sized by duration. Visual-search phases tend to spread
the fixation cloud across the field of view; counting and change-
detection phases cluster fixations near the relevant objects.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(yaw, pitch, color="lightgray", linewidth=0.4, label="raw gaze", zorder=1)

phase_colors = {
    "FreeExploration":  "#1f77b4",
    "VisualSearch":     "#ff7f0e",
    "CountingTask":     "#2ca02c",
    "ChangeDetection":  "#d62728",
}
for ph in observed_phases:
    mask = phase_per_fix == ph
    if not mask.any():
        continue
    fx = [fixs[i].centroid_x for i in range(len(fixs)) if mask[i]]
    fy = [fixs[i].centroid_y for i in range(len(fixs)) if mask[i]]
    sizes = [max(20, min(400, fixs[i].duration * 1000)) for i in range(len(fixs)) if mask[i]]
    ax.scatter(fx, fy, s=sizes, alpha=0.55, edgecolor="black", linewidths=0.3,
               c=phase_colors.get(ph, "#888"), label=ph, zorder=2)

ax.set_xlabel("yaw (deg)")
ax.set_ylabel("pitch (deg)")
ax.invert_yaxis()
ax.set_aspect("equal", adjustable="datalim")
ax.set_title(f"Scanpath by phase — {ctx.csv_path.name}")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

## 6. K-coefficient — sliding K(t) with pooled stats

Krejtz K = mean of (z(a) − z(d)) over fixation/saccade pairs. Pooled
(μ, σ) is computed over **all** session pairs; each window then averages
K_i over the pairs whose fixation start falls inside it. Sign tracks
the attention regime relative to the session baseline (positive ⇒
focal pattern, negative ⇒ ambient pattern under the `z(a)−z(d)`
convention; absolute sign is convention-dependent — the literature
splits — what matters is that the sign **changes** across phases of
different attentional demand).

`calculate_per_window()` returns one `KCoefficientResult` per window.

In [ ]:
calc = ela.KCoefficientCalculator()
windows = calc.calculate_per_window(fixs, saccs, window_seconds=5.0, step_seconds=1.0)

if not windows:
    print("Not enough fix/sac pairs for a sliding K(t).")
else:
    centers = np.array([(w.window_start + w.window_end) / 2 for w in windows])
    k_vals  = np.array([w.k_coefficient for w in windows])
    print(f"K(t): {len(windows)} windows, "
          f"K range [{k_vals.min():+.2f}, {k_vals.max():+.2f}], "
          f"median {np.median(k_vals):+.2f}")

    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.axhline(0, color="black", linewidth=0.5)
    ax.plot(centers, k_vals, linewidth=1.2)
    ax.fill_between(centers, 0, k_vals, where=k_vals>=0, alpha=0.25, label="K > 0")
    ax.fill_between(centers, 0, k_vals, where=k_vals<0,  alpha=0.25, color="#d62728", label="K < 0")

    # Shade phase boundaries if available.
    if phase_col is not None:
        cur_phase = None; cur_start = None
        for i in range(len(phase_col)):
            ph = phase_col[i]
            if ph != cur_phase:
                if cur_phase in phase_colors:
                    ax.axvspan(cur_start, ts[i], color=phase_colors[cur_phase], alpha=0.07, zorder=0)
                cur_phase, cur_start = ph, ts[i]
        if cur_phase in phase_colors:
            ax.axvspan(cur_start, ts[-1], color=phase_colors[cur_phase], alpha=0.07, zorder=0)

    ax.set_xlabel("time (s)")
    ax.set_ylabel("K (z-units)")
    ax.set_title("Sliding-window K(t) — 5 s window, 1 s step  (shaded by phase)")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()

## 7. Per-phase K against session-pooled (μ, σ)

This is the canonical Krejtz reporting unit: one K per condition,
referenced to the session-wide pool. A focal-leaning phase
(long fixations, short saccades) lands one side of zero; an
ambient-leaning phase lands the other. Sign is convention-dependent
(see §6) but **differences between phases are interpretable**.

In [ ]:
if n_pairs < 2:
    print("Not enough fix/sac pairs for K.")
else:
    durs_all = np.array([f.duration  for f in fixs[:n_pairs]])
    amps_all = np.array([s.amplitude for s in saccs[:n_pairs]])
    pooled = (
        float(durs_all.mean()), float(durs_all.std(ddof=1)),
        float(amps_all.mean()), float(amps_all.std(ddof=1)),
    )
    print(f"Session pool: n_pairs={n_pairs}, "
          f"mean_d={pooled[0]:.3f}s, mean_a={pooled[2]:.2f}deg")

    rows = []
    for ph in observed_phases:
        ph_mask = np.array([phase_of(fixs[i].start_time) == ph for i in range(n_pairs)])
        if int(ph_mask.sum()) < 2:
            continue
        fix_ph = [fixs[i] for i in range(n_pairs) if ph_mask[i]]
        sac_ph = [saccs[i] for i in range(n_pairs) if ph_mask[i]]
        r = calc.calculate(fix_ph, sac_ph, pooled_stats=pooled)
        # `r.mean_fixation_duration` and `mean_saccade_amplitude` echo the
        # pooled inputs (since we passed pooled_stats); recompute the per-
        # phase means here so the table reflects phase behaviour.
        ph_mean_fix = float(np.mean([f.duration  for f in fix_ph]))
        ph_mean_sac = float(np.mean([s.amplitude for s in sac_ph]))
        rows.append({
            "phase":           ph,
            "n_pairs":         int(ph_mask.sum()),
            "mean_fix_s":      round(ph_mean_fix, 3),
            "mean_sac_deg":    round(ph_mean_sac, 2),
            "K":               round(r.k_coefficient, 3),
            "attention_type":  r.attention_type.name,
        })

    if not rows:
        print("No SampleExperiment phase passed the >=2 pairs gate. "
              "Per-phase K and the bar chart are SampleExperiment-only "
              "displays; the sliding K(t) in §6 covers calibrator-only "
              "or other custom recordings.")
    else:
        per_phase_K = pd.DataFrame(rows)
        display(per_phase_K)

        fig, ax = plt.subplots(figsize=(7, 3.2))
        bars = ax.bar(per_phase_K["phase"], per_phase_K["K"],
                      color=[phase_colors.get(p, "#888") for p in per_phase_K["phase"]])
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_ylabel("K (z-units)")
        ax.set_title("Per-phase K vs. session pool")
        for bar, k in zip(bars, per_phase_K["K"]):
            ax.text(bar.get_x() + bar.get_width()/2, k,
                    f"{k:+.2f}", ha="center",
                    va="bottom" if k >= 0 else "top", fontsize=9)
        plt.xticks(rotation=15, ha="right")
        plt.tight_layout()
        plt.show()

## What next

- Notebook `06_sample_experiment` walks the same recording through the
  packaged `analyze_sample_experiment()` helper, which produces a tidy
  per-phase DataFrame with fixations, scanpath, gaze entropy, and
  LHIPA in one call.
- Notebook `05_pupil_lhipa` does the analogous per-phase split for
  pupil-based cognitive-load metrics (LHIPA and RIPA2).